# 01 - Descarga automática de fuentes a Parquet en Unity Catalog Volume

Este notebook descarga automáticamente datasets desde Hugging Face y Kaggle, los guarda como Parquet en un volumen de Unity Catalog y genera un dataset canónico combinado para el flujo Bronze/Silver/Gold.

Corrección aplicada: Kaggle ya no se importa al inicio del notebook. Primero se leen las credenciales desde widgets o secretos, luego se configuran las variables de entorno `KAGGLE_USERNAME` y `KAGGLE_KEY`, y solo después se importa/autentica `KaggleApi`. Esto evita el error `Could not find kaggle.json`.

Salidas principales:

- Raw Parquet por fuente y split: `/Volumes/<catalog>/<schema>/<volume>/sources/raw/...`
- Parquet canónico por dataset: `/Volumes/<catalog>/<schema>/<volume>/sources/processed/canonical/...`
- Dataset combinado: `/Volumes/<catalog>/<schema>/<volume>/sources/processed/combined/financial_sentiment_all`
- Manifiesto de auditoría: `<catalog>.<schema>.source_dataset_manifest`


In [0]:
%pip uninstall -y kagglehub kagglesdk
%pip install -q --force-reinstall "numpy<2" "datasets==4.6.1" "huggingface_hub>=0.29,<1.0" "kaggle==1.6.17" "fsspec>=2024.2.0" "openpyxl>=3.1.0"

# COMMAND ----------

dbutils.library.restartPython()

In [0]:
from datetime import datetime
from html import unescape
from pathlib import Path
import os
import re
import shutil
import unicodedata
import zipfile

import pandas as pd
from pyspark.sql import functions as F


In [0]:
# Widgets del Job / notebook
# Nota: si el widget ya existe, Databricks lanza excepción; por eso se crea de forma segura.

def ensure_text_widget(name: str, default: str):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default)


for widget_name, default_value in [
    ("catalog", "workspace"),
    ("schema", "financial_sentiment"),
    ("volume", "raw_landing"),
    ("enable_huggingface", "true"),
    ("enable_kaggle", "true"),
    ("force_refresh", "false"),
    ("fail_if_no_sources", "true"),
    ("fail_if_any_source_fails", "false"),
    ("kaggle_username", ""),
    ("kaggle_key", ""),
    ("kaggle_secret_scope", ""),
    ("kaggle_username_secret_key", ""),
    ("kaggle_key_secret_key", ""),
    ("hf_token", ""),
    # Importante: el cache de Hugging Face queda en una ruta temporal del driver.
    # Los Parquet raw/canonical sí quedan persistidos en el volumen de Unity Catalog.
    # Usar /Volumes como cache dispara un bug del runtime que intenta escribir en /local_disk0/<dataset> sin permisos.
    ("hf_driver_cache_root", "/tmp/huggingface_cache"),
]:
    ensure_text_widget(widget_name, default_value)


def widget_value(name: str, default: str = "") -> str:
    try:
        value = dbutils.widgets.get(name)
        if value is None:
            return default
        return str(value).strip()
    except Exception:
        return default


def to_bool(value: str, default: bool = False) -> bool:
    if value is None:
        return default
    value = str(value).strip().lower()
    if value in {"true", "1", "yes", "y", "si", "sí"}:
        return True
    if value in {"false", "0", "no", "n"}:
        return False
    return default


def try_get_secret(scope: str, key: str, default: str = "") -> str:
    if not scope or not key:
        return default
    try:
        return dbutils.secrets.get(scope=scope, key=key)
    except Exception as exc:
        print(f"No se pudo leer el secreto scope={scope}, key={key}. Detalle: {exc}")
        return default


catalog = widget_value("catalog", "workspace")
schema = widget_value("schema", "financial_sentiment")
volume = widget_value("volume", "raw_landing")

enable_huggingface = to_bool(widget_value("enable_huggingface", "true"), True)
enable_kaggle = to_bool(widget_value("enable_kaggle", "true"), True)
force_refresh = to_bool(widget_value("force_refresh", "false"), False)
fail_if_no_sources = to_bool(widget_value("fail_if_no_sources", "true"), True)
fail_if_any_source_fails = to_bool(widget_value("fail_if_any_source_fails", "false"), False)

hf_token = widget_value("hf_token", "") or None
hf_driver_cache_root = widget_value("hf_driver_cache_root", "/tmp/huggingface_cache") or "/tmp/huggingface_cache"

kaggle_secret_scope = widget_value("kaggle_secret_scope", "")
kaggle_username_secret_key = widget_value("kaggle_username_secret_key", "")
kaggle_key_secret_key = widget_value("kaggle_key_secret_key", "")

kaggle_username = widget_value("kaggle_username", "") or try_get_secret(
    kaggle_secret_scope, kaggle_username_secret_key, ""
)
kaggle_key = widget_value("kaggle_key", "") or try_get_secret(
    kaggle_secret_scope, kaggle_key_secret_key, ""
)

# Rutas persistentes del volumen. Aquí quedan los Parquet que consume Bronze.
volume_root = f"/Volumes/{catalog}/{schema}/{volume}"
raw_root = f"{volume_root}/sources/raw"
canonical_root = f"{volume_root}/sources/processed/canonical"
combined_root = f"{volume_root}/sources/processed/combined"
manifest_root = f"{volume_root}/sources/manifests"
kaggle_download_root = f"{volume_root}/sources/raw/kaggle_downloads"

# Cache temporal del driver para Hugging Face.
# No se usa /Volumes como cache para evitar PermissionError sobre /local_disk0/<dataset_key>.
hf_cache_path = f"{hf_driver_cache_root.rstrip('/')}/{catalog}_{schema}_{volume}"
hf_datasets_cache_path = f"{hf_cache_path}/datasets"
hf_home_path = f"{hf_cache_path}/home"
hf_hub_cache_path = f"{hf_cache_path}/hub"

for path in [
    raw_root,
    canonical_root,
    combined_root,
    manifest_root,
    kaggle_download_root,
]:
    Path(path).mkdir(parents=True, exist_ok=True)

for local_path in [hf_cache_path, hf_datasets_cache_path, hf_home_path, hf_hub_cache_path]:
    Path(local_path).mkdir(parents=True, exist_ok=True)

# Variables de entorno de Hugging Face apuntando a rutas escribibles del driver.
os.environ["HF_HOME"] = hf_home_path
os.environ["HF_HUB_CACHE"] = hf_hub_cache_path
os.environ["HF_DATASETS_CACHE"] = hf_datasets_cache_path
os.environ["TMPDIR"] = "/tmp"

# Kaggle: se configura por variables de entorno ANTES de importar KaggleApi.
if kaggle_username and kaggle_key:
    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key
else:
    print(
        "Kaggle está habilitado, pero no se recibieron credenciales. "
        "El notebook seguirá con Hugging Face si está habilitado. "
        "Para Kaggle, define los widgets kaggle_username y kaggle_key o usa secretos."
    )

download_batch_id = datetime.utcnow().strftime("%Y%m%d%H%M%S")
manifest_table = f"{catalog}.{schema}.source_dataset_manifest"
run_log_table = f"{catalog}.{schema}.pipeline_run_log"

print(f"catalog={catalog}")
print(f"schema={schema}")
print(f"volume={volume}")
print(f"volume_root={volume_root}")
print(f"hf_cache_path={hf_cache_path}")
print(f"enable_huggingface={enable_huggingface}")
print(f"enable_kaggle={enable_kaggle}")
print(f"force_refresh={force_refresh}")
print(f"download_batch_id={download_batch_id}")


In [0]:
# Creación defensiva de objetos de Unity Catalog usados por el notebook.
# Si 00_bootstrap_uc.py ya los creó, estas sentencias no cambian nada.

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume}`")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog}`.`{schema}`.`source_dataset_manifest` (
    download_batch_id STRING,
    dataset_key STRING,
    dataset_id STRING,
    source_platform STRING,
    status STRING,
    split_name STRING,
    rows_written BIGINT,
    raw_parquet_path STRING,
    canonical_parquet_path STRING,
    error_message STRING,
    event_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog}`.`{schema}`.`pipeline_run_log` (
    run_id STRING,
    step STRING,
    status STRING,
    message STRING,
    rows_written BIGINT,
    event_ts TIMESTAMP
)
USING DELTA
""")


In [0]:
# Imports que dependen de librerías instaladas.
# Importante: KaggleApi NO se importa aquí. Se importa dentro de get_kaggle_api(), luego de configurar variables de entorno.

import datasets
from datasets import load_dataset

try:
    datasets.utils.logging.disable_progress_bar()
except Exception:
    pass

POSITIVE_LABELS = {"positive", "mildly positive", "moderately positive", "strong positive", "bullish", "pos", "1", "positivo"}
NEGATIVE_LABELS = {"negative", "mildly negative", "moderately negative", "strong negative", "bearish", "neg", "-1", "negativo"}
NEUTRAL_LABELS = {"neutral", "neu", "0", "neutro"}
BASE_COLUMNS = ["dataset_id", "dataset_label", "source_platform", "split", "text", "label_normalized"]


def clean_text(value):
    if pd.isna(value):
        return None
    text = unescape(str(value))
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8", "ignore")
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text or None


def normalize_label(value):
    if pd.isna(value):
        return None
    normalized = str(value).strip().lower()
    if normalized in POSITIVE_LABELS:
        return "positive"
    if normalized in NEGATIVE_LABELS:
        return "negative"
    if normalized in NEUTRAL_LABELS:
        return "neutral"
    return normalized


def safe_dataset_key(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]+", "_", value).strip("_").lower()


def find_first_existing_column(df: pd.DataFrame, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        key = str(candidate).strip().lower()
        if key in lower_map:
            return lower_map[key]
    return None


def write_spark_parquet_from_pandas(pdf: pd.DataFrame, path: str, mode: str = "overwrite") -> int:
    if len(pdf) == 0:
        raise ValueError(f"No se escribirá Parquet vacío en {path}")
    
    # Convert DataFrame to dict and back to ensure native pandas dtypes (avoids pyarrow issues)
    pdf_native = pd.DataFrame(pdf.to_dict('records'))
    
    sdf = spark.createDataFrame(pdf_native)
    sdf.write.mode(mode).format("parquet").save(path)
    return int(len(pdf))


def compute_split_counts(total_rows):
    test_rows = round(total_rows * 0.2)
    validation_rows = round(total_rows * 0.2)
    train_rows = total_rows - test_rows - validation_rows
    if train_rows < 0:
        validation_rows = max(validation_rows + train_rows, 0)
        train_rows = total_rows - test_rows - validation_rows
    if train_rows < 0:
        test_rows = max(test_rows + train_rows, 0)
        train_rows = total_rows - test_rows - validation_rows
    return {"train": train_rows, "validation": validation_rows, "test": test_rows}


def derive_curated_splits(raw_split_frames):
    if not raw_split_frames:
        raise ValueError("No hay splits para construir dataset curado.")
    ordered_names = [s for s in ["train", "validation", "test"] if s in raw_split_frames]
    ordered_names += [s for s in raw_split_frames.keys() if s not in ordered_names]
    raw_df = pd.concat([raw_split_frames[s] for s in ordered_names], ignore_index=True, sort=False)
    raw_df = raw_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    counts = compute_split_counts(len(raw_df))
    train_end = counts["train"]
    validation_end = train_end + counts["validation"]
    return {
        "train": raw_df.iloc[:train_end].reset_index(drop=True),
        "validation": raw_df.iloc[train_end:validation_end].reset_index(drop=True),
        "test": raw_df.iloc[validation_end:].reset_index(drop=True),
    }


def standardize_lwrf(df, split_name):
    out = df.copy()
    text_col = find_first_existing_column(out, ["input", "text", "sentence"])
    label_col = find_first_existing_column(out, ["output", "label", "sentiment"])
    if text_col is None or label_col is None:
        raise ValueError(f"Columnas no compatibles para lwrf42. Columnas: {list(out.columns)}")
    out["dataset_id"] = "lwrf42/financial-sentiment-dataset"
    out["dataset_label"] = "lwrf42_financial_sentiment_dataset"
    out["source_platform"] = "huggingface"
    out["split"] = split_name
    out["text"] = out[text_col].map(clean_text)
    out["label_normalized"] = out[label_col].map(normalize_label)
    return out[BASE_COLUMNS].copy()


def standardize_kenpache_english(df, split_name):
    out = df.copy()
    lang_col = find_first_existing_column(out, ["language", "lang"])
    if lang_col is not None:
        out = out[out[lang_col].astype(str).str.lower().eq("en")].copy()
    text_col = find_first_existing_column(out, ["sentence", "text", "headline"])
    label_col = find_first_existing_column(out, ["label", "sentiment"])
    if text_col is None or label_col is None:
        raise ValueError(f"Columnas no compatibles para Kenpache. Columnas: {list(out.columns)}")
    out["dataset_id"] = "Kenpache/multilingual-financial-sentiment"
    out["dataset_label"] = "kenpache_multilingual_financial_sentiment_en"
    out["source_platform"] = "huggingface"
    out["split"] = split_name
    out["text"] = out[text_col].map(clean_text)
    out["label_normalized"] = out[label_col].map(normalize_label)
    return out[BASE_COLUMNS].copy()


def standardize_maguid(df, split_name):
    out = df.copy()
    text_col = find_first_existing_column(out, ["text", "sentence", "headline"])
    label_col = find_first_existing_column(out, ["polarity", "label", "sentiment"])
    if text_col is None or label_col is None:
        raise ValueError(f"Columnas no compatibles para maguid28. Columnas: {list(out.columns)}")
    out["dataset_id"] = "maguid28/combined_financial_phrasebank_twitter_news_sentiment"
    out["dataset_label"] = "maguid28_combined_financial_phrasebank_twitter_news_sentiment"
    out["source_platform"] = "huggingface"
    out["split"] = split_name
    out["text"] = out[text_col].map(clean_text)
    out["label_normalized"] = out[label_col].map(normalize_label)
    return out[BASE_COLUMNS].copy()


def standardize_kaggle(df, split_name):
    out = df.copy()
    text_col = find_first_existing_column(out, ["sentence", "text", "headline", "news", "content"])
    label_col = find_first_existing_column(out, ["sentiment", "label", "polarity", "target"])
    if text_col is None or label_col is None:
        raise ValueError(f"Columnas no compatibles para Kaggle. Columnas: {list(out.columns)}")
    out["dataset_id"] = "sbhatti/financial-sentiment-analysis"
    out["dataset_label"] = "sbhatti_financial_sentiment_analysis"
    out["source_platform"] = "kaggle"
    out["split"] = split_name
    out["text"] = out[text_col].map(clean_text)
    out["label_normalized"] = out[label_col].map(normalize_label)
    return out[BASE_COLUMNS].copy()

In [0]:
source_configs = [
    {
        "key": "lwrf42_financial_sentiment_dataset",
        "repo_id": "lwrf42/financial-sentiment-dataset",
        "source_platform": "huggingface",
        "standardizer": standardize_lwrf,
        "enabled": enable_huggingface,
    },
    {
        "key": "kenpache_multilingual_financial_sentiment",
        "repo_id": "Kenpache/multilingual-financial-sentiment",
        "source_platform": "huggingface",
        "standardizer": standardize_kenpache_english,
        "enabled": enable_huggingface,
    },
    {
        "key": "maguid28_combined_financial_phrasebank_twitter_news_sentiment",
        "repo_id": "maguid28/combined_financial_phrasebank_twitter_news_sentiment",
        "source_platform": "huggingface",
        "standardizer": standardize_maguid,
        "enabled": enable_huggingface,
    },
    {
        "key": "sbhatti_financial_sentiment_analysis",
        "repo_id": "sbhatti/financial-sentiment-analysis",
        "source_platform": "kaggle",
        "standardizer": standardize_kaggle,
        "enabled": enable_kaggle,
        "default_split": "train",
    },
]

manifest_rows = []
combined_frames = []
failed_sources = []


def append_manifest(config, status, split_name="all", rows_written=0, raw_parquet_path=None, canonical_parquet_path=None, error_message=None):
    manifest_rows.append({
        "download_batch_id": download_batch_id,
        "dataset_key": config["key"],
        "dataset_id": config["repo_id"],
        "source_platform": config["source_platform"],
        "status": status,
        "split_name": split_name,
        "rows_written": int(rows_written or 0),
        "raw_parquet_path": raw_parquet_path,
        "canonical_parquet_path": canonical_parquet_path,
        "error_message": str(error_message) if error_message else None,
        "event_ts": datetime.utcnow(),
    })


def load_huggingface_raw_splits(config):
    """Descarga Hugging Face usando cache local escribible y luego el pipeline guarda Parquet en el volumen."""
    dataset_key = safe_dataset_key(config["key"])
    dataset_cache_dir = f"{hf_datasets_cache_path}/{dataset_key}"

    if force_refresh and Path(dataset_cache_dir).exists():
        shutil.rmtree(dataset_cache_dir, ignore_errors=True)

    Path(dataset_cache_dir).mkdir(parents=True, exist_ok=True)

    print(f"Cache Hugging Face driver: {dataset_cache_dir}")
    ds = load_dataset(
        config["repo_id"],
        cache_dir=dataset_cache_dir,
        token=hf_token,
        verification_mode="no_checks",
        download_mode="force_redownload" if force_refresh else "reuse_dataset_if_exists",
    )
    
    # Convert to pandas with native dtypes (not pyarrow-backed)
    result = {}
    for split_name, split_ds in ds.items():
        # Use dict conversion to avoid pyarrow ChunkedArray issues
        data_dict = split_ds[:]
        df = pd.DataFrame(data_dict)
        # Force conversion to native pandas dtypes
        for col in df.columns:
            if hasattr(df[col].dtype, '__arrow_c_schema__'):
                df[col] = df[col].astype('object')
        result[split_name] = df
    return result


def get_kaggle_api():
    """Autentica Kaggle solo cuando se va a usar. Evita error de kaggle.json en el import."""
    if not kaggle_username or not kaggle_key:
        raise ValueError(
            "Faltan credenciales de Kaggle. Define kaggle_username y kaggle_key como parámetros del Job, "
            "o usa kaggle_secret_scope + kaggle_username_secret_key + kaggle_key_secret_key."
        )

    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key

    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    return api


def load_kaggle_raw_splits(config):
    api = get_kaggle_api()
    dataset_key = safe_dataset_key(config["key"])
    dataset_root = Path(kaggle_download_root) / dataset_key
    extract_root = dataset_root / "extracted"

    if force_refresh and dataset_root.exists():
        dbutils.fs.rm(str(dataset_root), True)

    dataset_root.mkdir(parents=True, exist_ok=True)
    extract_root.mkdir(parents=True, exist_ok=True)

    print(f"Descargando Kaggle dataset: {config['repo_id']} en {dataset_root}")
    api.dataset_download_files(
        config["repo_id"],
        path=str(dataset_root),
        unzip=False,
        quiet=False,
    )

    zip_files = sorted(dataset_root.glob("*.zip"))
    if not zip_files:
        raise FileNotFoundError(f"No se encontró ZIP de Kaggle en {dataset_root}")

    for zip_file in zip_files:
        with zipfile.ZipFile(zip_file, "r") as zf:
            zf.extractall(extract_root)

    dataframes = []
    for file_path in sorted(extract_root.rglob("*")):
        suffix = file_path.suffix.lower()
        try:
            if suffix == ".csv":
                df = pd.read_csv(file_path)
            elif suffix in {".xlsx", ".xls"}:
                df = pd.read_excel(file_path)
            else:
                continue
        except Exception as exc:
            print(f"Archivo omitido por error de lectura: {file_path}. Detalle: {exc}")
            continue

        # Validación mínima: debe tener alguna columna de texto y etiqueta.
        text_col = find_first_existing_column(df, ["sentence", "text", "headline", "news", "content"])
        label_col = find_first_existing_column(df, ["sentiment", "label", "polarity", "target"])
        if text_col is None or label_col is None:
            print(f"Archivo omitido por columnas no compatibles: {file_path}. Columnas: {list(df.columns)}")
            continue

        df["source_file"] = str(file_path)
        dataframes.append(df)

    if not dataframes:
        raise ValueError(f"No se encontraron archivos Kaggle válidos para {config['repo_id']}")

    return {config.get("default_split", "train"): pd.concat(dataframes, ignore_index=True, sort=False)}


def process_source(config):
    if not config["enabled"]:
        append_manifest(config, "skipped", error_message="Fuente deshabilitada por parámetro del job")
        return

    print("=" * 100)
    print(f"Procesando fuente: {config['repo_id']} [{config['source_platform']}]")

    if config["source_platform"] == "huggingface":
        raw_splits = load_huggingface_raw_splits(config)
    elif config["source_platform"] == "kaggle":
        raw_splits = load_kaggle_raw_splits(config)
    else:
        raise ValueError(f"source_platform no soportado: {config['source_platform']}")

    # 1) Guardar raw parquet por split en el volumen.
    for split_name, raw_pdf in raw_splits.items():
        raw_path = f"{raw_root}/{config['source_platform']}/{config['key']}/{split_name}"
        rows = write_spark_parquet_from_pandas(raw_pdf, raw_path, mode="overwrite")
        append_manifest(config, "raw_parquet_saved", split_name, rows, raw_parquet_path=raw_path)
        print(f"Raw Parquet guardado: {raw_path} filas={rows}")

    # 2) Derivar splits train/validation/test comparables y estandarizar columnas.
    curated_splits = derive_curated_splits(raw_splits)
    canonical_frames = []
    for split_name, split_pdf in curated_splits.items():
        canonical_split = config["standardizer"](split_pdf, split_name)
        canonical_split = canonical_split[
            canonical_split["text"].notna() & canonical_split["label_normalized"].notna()
        ].copy()
        canonical_split = canonical_split[canonical_split["text"].astype(str).str.len() > 0].copy()
        canonical_split["download_batch_id"] = download_batch_id
        canonical_split["source_record_hash"] = pd.util.hash_pandas_object(
            canonical_split[["dataset_label", "split", "text", "label_normalized"]].astype(str),
            index=False,
        ).astype(str)
        canonical_frames.append(canonical_split)

    if not canonical_frames:
        raise ValueError(f"La fuente {config['repo_id']} no generó registros canónicos válidos.")

    canonical_pdf = pd.concat(canonical_frames, ignore_index=True, sort=False)
    canonical_pdf = canonical_pdf.drop_duplicates(subset=["source_record_hash"])
    canonical_path = f"{canonical_root}/{config['key']}"
    rows = write_spark_parquet_from_pandas(canonical_pdf, canonical_path, mode="overwrite")
    append_manifest(config, "canonical_parquet_saved", "all", rows, canonical_parquet_path=canonical_path)
    combined_frames.append(canonical_pdf)
    print(f"Canonical Parquet guardado: {canonical_path} filas={rows}")

In [0]:
for config in source_configs:
    try:
        process_source(config)
    except Exception as exc:
        failed_sources.append((config["key"], str(exc)))
        append_manifest(config, "failed", config.get("default_split", "all"), error_message=str(exc))
        print(f"Falló fuente {config['repo_id']}: {exc}")

successful_frames = [df for df in combined_frames if len(df) > 0]
if successful_frames:
    combined_pdf = pd.concat(successful_frames, ignore_index=True, sort=False)
    combined_pdf = combined_pdf.drop_duplicates(subset=["source_record_hash"])
    combined_path = f"{combined_root}/financial_sentiment_all"
    combined_rows = write_spark_parquet_from_pandas(combined_pdf, combined_path, mode="overwrite")
    dbutils.fs.rm(f"{manifest_root}/latest_successful_combined_path.txt", True)
    dbutils.fs.put(f"{manifest_root}/latest_successful_combined_path.txt", combined_path, True)
    print(f"Dataset combinado guardado: {combined_path} filas={combined_rows}")
else:
    combined_rows = 0
    if fail_if_no_sources:
        raise RuntimeError(
            "No se logró descargar ninguna fuente. Revisa conectividad, credenciales de Kaggle o restricciones de Free Edition."
        )

manifest_pdf = pd.DataFrame(manifest_rows)
if len(manifest_pdf) > 0:
    spark.createDataFrame(manifest_pdf).write.mode("append").format("delta").saveAsTable(manifest_table)
    manifest_json_str = manifest_pdf.to_json(orient="records", date_format="iso", force_ascii=False)
    dbutils.fs.put(f"{manifest_root}/{download_batch_id}_manifest.json", manifest_json_str, True)

if failed_sources and fail_if_any_source_fails:
    raise RuntimeError(f"Fallaron fuentes requeridas: {failed_sources}")

status = "SUCCESS" if not failed_sources else "PARTIAL_SUCCESS"
message = (
    f"Descarga automática finalizada. "
    f"Fuentes exitosas={len(successful_frames)}, fallidas={len(failed_sources)}, filas combinadas={combined_rows}."
)
spark.createDataFrame([
    (download_batch_id, "01_download_source_parquets", status, message, int(combined_rows), datetime.utcnow())
], "run_id string, step string, status string, message string, rows_written long, event_ts timestamp").write.mode("append").saveAsTable(run_log_table)

try:
    dbutils.jobs.taskValues.set(key="download_batch_id", value=download_batch_id)
    dbutils.jobs.taskValues.set(key="canonical_root", value=canonical_root)
    dbutils.jobs.taskValues.set(key="combined_rows", value=str(combined_rows))
except Exception:
    pass

display(spark.table(manifest_table).where(F.col("download_batch_id") == download_batch_id))
print(message)